# Final ML Coding Interview Practice Notebook — With Tests

This version has TODOs plus executable tests after each implementation. Implement a section, run its test, debug, then continue.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 1. Token log-prob gather

Given logits `[B,T,V]` and chosen IDs `[B,T]`, return `[B,T]`.

In [ ]:
def gather_token_logprobs(logits: torch.Tensor, token_ids: torch.Tensor):
    # TODO: implement
    pass

In [3]:
B,T,V=2,3,5
logits=torch.randn(B,T,V)
ids=torch.tensor([[0,2,4],[1,3,0]])
out=gather_token_logprobs(logits,ids)
expected=F.log_softmax(logits,-1).gather(-1,ids.unsqueeze(-1)).squeeze(-1)
assert out.shape==(B,T)
assert torch.allclose(out,expected,atol=1e-6)
print("✓ token log-prob gather")

✓ token log-prob gather


## 2. Sequence log probability

LM shift: logits at `t` predict token `t+1`. Return masked sum and mean per sequence.

In [ ]:
def sequence_logprob(logits,input_ids,attention_mask=None):
    # TODO: implement
    pass

In [9]:
B,T,V=2,4,6
logits=torch.randn(B,T,V)
ids=torch.tensor([[1,2,3,4],[2,1,5,0]])
mask=torch.tensor([[1,1,1,1],[1,1,1,0]],dtype=torch.bool)
sum_lp,mean_lp=sequence_logprob(logits,ids,mask)
shift_logits=logits[:,:-1]
targets=ids[:,1:]
target_mask=mask[:,1:]
expected_tok=F.log_softmax(shift_logits,-1).gather(-1,targets.unsqueeze(-1)).squeeze(-1)
expected_sum=(expected_tok*target_mask).sum(-1)
expected_mean=expected_sum/target_mask.sum(-1).clamp_min(1)
assert torch.allclose(sum_lp,expected_sum,atol=1e-6)
assert torch.allclose(mean_lp,expected_mean,atol=1e-6)
print("✓ sequence log-prob")

✓ sequence log-prob


## 3. REINFORCE

`L = - advantage * log_prob`. Support `[B]` or `[B,T]` advantages and optional mask.

In [ ]:
def reinforce_loss(token_logprobs,advantages,mask=None):
    # TODO: implement
    pass

In [25]:
lp=torch.tensor([[-1.,-2.],[-.5,-1.5]])
adv=torch.tensor([2.,-1.])
mask=torch.tensor([[1,1],[1,0]],dtype=torch.bool)
loss=reinforce_loss(lp,adv,mask)
expected=-(lp*adv[:,None])
expected=expected[mask].mean()
assert torch.allclose(loss,expected)
print("✓ REINFORCE")

✓ REINFORCE


In [23]:
loss

tensor([ 3.0000, -0.5000])

## 4. PPO clipped surrogate

`ratio = exp(new_logp-old_logp)` and `-min(ratio*A, clipped_ratio*A)`

In [ ]:
def ppo_clipped_policy_loss(new_logprobs,old_logprobs,advantages,clip_eps=0.2,mask=None):
    # TODO: implement
    pass

In [37]:
old=torch.log(torch.tensor([1.,1.,1.]))
new=torch.log(torch.tensor([1.5,.5,1.1]))
adv=torch.tensor([1.,1.,-1.])
loss=ppo_clipped_policy_loss(new,old,adv,0.2)
ratio=torch.exp(new-old)
expected=-torch.minimum(ratio*adv,ratio.clamp(.8,1.2)*adv).mean()
assert torch.allclose(loss,expected)
print("✓ PPO clipped loss")

✓ PPO clipped loss


## 5. PPO value-function loss

Basic: `0.5*(V-return)^2`. Bonus: clipped value loss.

In [ ]:
def value_loss(values, returns, old_values=None, clip_eps=0.1):
    # TODO: implement
    pass

In [43]:
v=torch.tensor([1.,3.]); r=torch.tensor([2.,1.])
basic=value_loss(v,r)
assert torch.allclose(basic,0.5*((v-r)**2).mean())
ov=torch.tensor([0.,2.])
cl=value_loss(v,r,ov,.5)
vc=ov+(v-ov).clamp(-.5,.5)
expected=.5*torch.maximum((v-r)**2,(vc-r)**2).mean()
assert torch.allclose(cl,expected)
print("✓ value loss")

✓ value loss


## 6. Generalized Advantage Estimation

`delta_t=r_t+gamma*V_{t+1}(1-done)-V_t`; `A_t=delta_t+gamma*lambda*(1-done)*A_{t+1}`.

In [ ]:
def compute_gae(rewards,values,dones,gamma=0.99,lam=0.95):
    # rewards [B,T], values [B,T+1], dones [B,T]
    # TODO: implement
    pass

In [49]:
rewards=torch.tensor([[1.,1.]])
values=torch.tensor([[.5,.25,0.]])
dones=torch.tensor([[0.,1.]])
adv,ret=compute_gae(rewards,values,dones,gamma=1.,lam=1.)
expected_adv=torch.tensor([[1.5,.75]])
assert torch.allclose(adv,expected_adv)
assert torch.allclose(ret,expected_adv+values[:,:-1])
print("✓ GAE")

✓ GAE


## 7. GRPO group advantage normalization

Normalize rewards `[B,G]` within each group.

In [ ]:
def grpo_group_advantages(rewards: torch.Tensor,eps=1e-8):
    # TODO: implement
    pass

In [57]:
r=torch.tensor([[1.,2.,3.],[5.,5.,5.]])
a=grpo_group_advantages(r)
assert a.shape==r.shape
assert torch.allclose(a[0].mean(),torch.tensor(0.),atol=1e-6)
assert torch.allclose(a[1],torch.zeros_like(a[1]),atol=1e-5)
print("✓ GRPO group advantages")

torch.Size([2, 3])
✓ GRPO group advantages


## 8. GRPO clipped surrogate

Same clipping idea as PPO, using group-derived advantages.

In [ ]:
def grpo_clipped_loss(new_logprobs,old_logprobs,advantages,clip_eps=0.2,mask=None):
    # new/old logprobs: [B,G,T]
    # advantages: [B,G] or [B,G,T]
    
    # TODO: implement
    pass

In [167]:
# DAPO
# dapo_loss = masked_losses.sum() / mask.sum().clamp(min=1.0)


In [166]:
new=torch.log(torch.tensor([[[1.2,.8],[1.1,.9]]]))
old=torch.zeros_like(new)
adv=torch.tensor([[1.,-1.]])
loss=grpo_clipped_loss(new,old,adv,.2)
r=torch.exp(new-old); a=adv.unsqueeze(-1)
expected=-torch.minimum(r*a,r.clamp(.8,1.2)*a).mean()
assert torch.allclose(loss,expected)
print("✓ GRPO clipped loss")

torch.Size([1, 2, 2])
✓ GRPO clipped loss


## 9. RLHF sampled KL penalty

Approximation: `log pi_policy(a)-log pi_ref(a)`.

In [ ]:
def sampled_kl_penalty(policy_logprobs,ref_logprobs,beta=0.1,mask=None):
    # TODO: implement
    pass

In [79]:
p=torch.tensor([[-1.,-2.],[-.5,-.7]])
q=torch.tensor([[-1.2,-1.8],[-.4,-1.]])
mask=torch.tensor([[1,1],[1,0]],dtype=torch.bool)
pen=sampled_kl_penalty(p,q,.5,mask)
expected=.5*(p-q)[mask].mean()
assert torch.allclose(pen,expected)
print("✓ sampled KL penalty")

tensor(-0.0167)
✓ sampled KL penalty


## 10. GRPO K3 estimator

With `x=log p-log q`, `k3=exp(-x)-1+x`.

In [ ]:
def k3_kl_estimator(policy_logprobs,ref_logprobs):
    # TODO: implement
    pass

In [81]:
p=torch.tensor([-1.,-2.,0.]); q=torch.tensor([-1.5,-1.5,-.2])
k3=k3_kl_estimator(p,q); x=p-q
expected=torch.exp(-x)-1+x
assert torch.allclose(k3,expected)
assert (k3>=-1e-7).all()
print("✓ K3 estimator")

✓ K3 estimator


## 11. Reward-model pairwise loss

`-log sigmoid(chosen-rejected)`

In [ ]:
def reward_model_pairwise_loss(chosen_scores,rejected_scores):
    # TODO: implement
    pass

In [89]:
c=torch.tensor([5.,2.]); r=torch.tensor([0.,1.])
loss=reward_model_pairwise_loss(c,r)
assert torch.allclose(loss,-F.logsigmoid(c-r).mean())
assert reward_model_pairwise_loss(torch.tensor([10.]),torch.tensor([-10.])) < reward_model_pairwise_loss(torch.tensor([0.]),torch.tensor([0.]))
print("✓ reward model loss")

✓ reward model loss


## 12. DPO implicit reward

`beta*(policy_logp-ref_logp)`

In [ ]:
def dpo_implicit_reward(policy_logp,ref_logp,beta=0.1):
    # Compute total Implicit Rewards by SUMMING across token dimensions (dim=-1) given to us as logp
    # policy_seq_logp = (policy_token_logp * mask).sum(-1)
    # ref_seq_logp = (ref_token_logp * mask).sum(-1)
    # TODO: implement
    pass

In [95]:
pi=torch.tensor([-2.,-1.]); ref=torch.tensor([-3.,-.5])
out=dpo_implicit_reward(pi,ref,.2)
assert torch.allclose(out,.2*(pi-ref))
print("✓ DPO implicit reward")

✓ DPO implicit reward


## 13. DPO loss

`-log sigmoid(beta * [(pi_c-pi_r)-(ref_c-ref_r)])`

In [ ]:
def dpo_loss(policy_chosen_logp,policy_rejected_logp,ref_chosen_logp,ref_rejected_logp,beta=0.1):
    # TODO: implement
    pass

In [99]:
pc=torch.tensor([-1.,-2.]); pr=torch.tensor([-2.,-2.5])
rc=torch.tensor([-1.5,-2.]); rr=torch.tensor([-1.7,-2.1])
loss=dpo_loss(pc,pr,rc,rr,.5)
z=.5*((pc-pr)-(rc-rr))
assert torch.allclose(loss,-F.logsigmoid(z).mean())
print("✓ DPO loss")

✓ DPO loss


## 14. Cosine LR with warmup

Linear warmup, then cosine decay.

In [ ]:
def cosine_lr(step,total_steps,warmup_steps,base_lr,min_lr=0.0):
    # TODO: implement
    pass

In [101]:
base=1e-3
assert abs(cosine_lr(0,100,10,base)-0.0)<1e-12
assert abs(cosine_lr(5,100,10,base)-.5*base)<1e-12
assert abs(cosine_lr(10,100,10,base)-base)<1e-12
assert base>cosine_lr(55,100,10,base)>cosine_lr(99,100,10,base)>=0
print("✓ cosine LR")

✓ cosine LR


## 15. Gradient accumulation

Divide microbatch loss by `accum_steps`; step only at accumulation boundaries.

In [ ]:
model=nn.Linear(8,3)
optimizer=torch.optim.SGD(model.parameters(),lr=.01)
accum_steps=4
optimizer.zero_grad()
num_optimizer_steps=0

# TODO: implement training loop
pass

In [106]:
assert num_optimizer_steps==2, num_optimizer_steps
print("✓ gradient accumulation")

✓ gradient accumulation


## 16. Mixed precision + unscaling

Correct order: autocast -> scale -> backward -> unscale -> clip -> step -> update.

In [ ]:
def run_amp_step():
    # TODO: implement
    pass

In [169]:
assert run_amp_step()
print("✓ AMP step")

✓ AMP step


## 17. Custom cross entropy

Implement from logits using stable log-softmax/logsumexp.

In [ ]:
def custom_cross_entropy(logits,targets):
    # TODO: implement
    pass

In [120]:
x=torch.randn(16,7)*3; y=torch.randint(0,7,(16,))
assert torch.allclose(custom_cross_entropy(x,y),F.cross_entropy(x,y),atol=1e-6)
print("✓ custom CE")

✓ custom CE


## 18. Custom BCE with logits

Stable identity: `max(x,0)-x*y+log(1+exp(-abs(x)))`.

In [ ]:
def custom_bce_with_logits(logits, targets):
    # Ensure targets are the same data type as logits (float)
    # TODO: implement
    pass

In [122]:
x=torch.tensor([-100.,-2.,0.,2.,100.]); y=torch.tensor([0.,1.,0.,1.,1.])
mine=custom_bce_with_logits(x,y); ref=F.binary_cross_entropy_with_logits(x,y)
assert torch.isfinite(mine)
assert torch.allclose(mine,ref,atol=1e-6)
print("✓ custom BCE")

✓ custom BCE


## 19. Linear regression from scratch

Do not use `nn.Linear`.

In [ ]:
torch.manual_seed(19)
N,D=128,4
X=torch.randn(N,D); true_w=torch.tensor([2.,-1.,.5,3.]); true_b=torch.tensor([.7])
y=X@true_w+true_b+.05*torch.randn(N)
w=torch.zeros(D,requires_grad=True); b=torch.zeros(1,requires_grad=True)
lr=.05

# TODO: implement training loop
pass

In [127]:
assert torch.norm(w.detach()-true_w)<.2
assert torch.abs(b.detach()-true_b)<.2
print("✓ linear regression")

✓ linear regression


## 20. Logistic regression from scratch

Use logits `X@w+b` and BCE-with-logits.

In [ ]:
torch.manual_seed(20)
N,D=300,5
X=torch.randn(N,D); true_w=torch.randn(D); true_b=torch.tensor([-.2])
y=((X@true_w+true_b)>0).float()
w=torch.zeros(D,requires_grad=True); b=torch.zeros(1,requires_grad=True)
lr=.1

# TODO: implement training loop
pass

In [129]:
with torch.no_grad():
    acc=(((X@w+b)>0).float()==y).float().mean().item()
assert acc>.95, acc
print("✓ logistic regression",acc)

✓ logistic regression 0.9900000095367432


## 21. Linear SVM

Hinge loss: `max(0,1-y*score)` with labels `{-1,+1}`.

The core objective is not just to get the prediction right, but to ensure that the correct class score dominates the wrong class by a safety margin of at least \(1\). If the model satisfies \(y \cdot s \geq 1\), it incurs zero loss. If it falls short, it pays a linear penalty. This gives rise to the Hinge Loss function:

In [ ]:
def linear_svm_loss(scores, labels_pm1, w=None, reg=0.0):
    """
    Computes the multiclass/binary margin-based hinge loss.
    
    Arguments:
    scores: PyTorch Tensor of shape (N,) containing the predicted values (w^T * x + b)
    labels_pm1: PyTorch Tensor of shape (N,) containing true labels encoded as +1 or -1
    w: PyTorch Tensor containing the model weights (used for regularization)
    reg: float, regularization strength
    
    Returns:
    loss: PyTorch scalar tensor containing the total loss
    """
    # 1. Compute the margin for all elements simultaneously: 1 - y * s
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    # TODO: implement
    pass

In [139]:
scores=torch.tensor([2.,.5,-2.,-.2]); labels=torch.tensor([1.,1.,-1.,-1.])
loss=linear_svm_loss(scores,labels)
assert torch.allclose(loss,torch.relu(1-labels*scores).mean())
wtest=torch.tensor([1.,2.])
assert linear_svm_loss(scores,labels,wtest,.1)>loss
print("✓ linear SVM")

✓ linear SVM


## 22. Custom SGD

Manual parameter update.

In [ ]:
def sgd_step(params,lr):
    # TODO: implement
    pass

In [131]:
p=torch.tensor([1.,2.],requires_grad=True); (p**2).sum().backward(); sgd_step([p],.1)
assert torch.allclose(p.detach(),torch.tensor([.8,1.6]))
print("✓ custom SGD")

✓ custom SGD


## 23. Custom Adam

Implement first/second moments, bias correction, and update.

In [ ]:
class MyAdam:
    def __init__(self,params,lr=1e-3,betas=(.9,.999),eps=1e-8):
        # TODO: implement
        pass
    @torch.no_grad()
    def step(self):
        # TODO: implement
        pass
    def zero_grad(self):
        # TODO: implement
        pass

In [ ]:
import torch
from torch.optim import Optimizer

class MyEasyAdam(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # Pack defaults to pass to the parent Optimizer class
        defaults = dict(lr=lr, betas=betas, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        """Performs a single optimization step."""
        # TODO: implement
        pass

In [136]:
p1=torch.tensor([1.,-2.],requires_grad=True); p2=p1.detach().clone().requires_grad_(True)
mine=MyAdam([p1],lr=1e-2); ref=torch.optim.Adam([p2],lr=1e-2)
for _ in range(5):
    (p1**2).sum().backward(); (p2**2).sum().backward(); mine.step(); ref.step(); mine.zero_grad(); ref.zero_grad()
assert torch.allclose(p1.detach(),p2.detach(),atol=1e-6)
print("✓ custom Adam")

✓ custom Adam


# 24. Parallelism rapid review

**DP:** model replicated, batch split, gradients synchronized.

**TP:** individual layer tensors/matmuls split; intra-layer communication.

**PP:** layers split into stages; microbatches create pipeline bubbles.

**ZeRO-1:** shard optimizer states.  
**ZeRO-2:** shard optimizer states + gradients.  
**ZeRO-3/FSDP full shard:** shard optimizer states + gradients + parameters.

**Activation checkpointing:** save activation memory by recomputing forward activations during backward.

## 25. Parallelism memory calculation

Assume params `P`, grads `P`, Adam moments `2P`; total `4P`.

In [ ]:
def rough_model_state_elements(P,world_size,mode):
    """
    Calculates the memory footprint of model states (Parameters, Gradients, 
    and Optimizer tracking states) per GPU measured in total element count.
    
    Assumes standard mixed-precision tracking (e.g., Adam):
      - 1 copy of fp16 Parameters (P elements)
      - 1 copy of fp16 Gradients (P elements)
      - 3 copies of fp32 Optimizer States (3*P elements total for fp32 master weights, 
        momentum vector, and variance vector).
    """
    # TODO: implement

    pass

In [149]:
P=100; W=4
assert rough_model_state_elements(P,W,"dp")==400
assert rough_model_state_elements(P,W,"zero1")==250
assert rough_model_state_elements(P,W,"zero2")==175
assert rough_model_state_elements(P,W,"zero3")==100
print("✓ ZeRO memory math")

✓ ZeRO memory math


# 26. Pandas brief review
Know `head`, `info`, `describe`, `isna`, `value_counts`, boolean filtering, `groupby`, `merge`, `sort_values`, `drop_duplicates`, `fillna`, `Series.map`, `Series.apply`, `DataFrame.apply`. Prefer vectorized operations when possible.

In [ ]:
df=pd.DataFrame({"user_id":[1,1,2,3,3],"score":[.2,.8,.4,.9,None],"label":[0,1,0,1,1]})
train_mean=.5
# TODO:
df['score'] = None  # TODO: implement
filled = None  # TODO: implement
mean_by_label = None  # TODO: implement
count_by_user = None  # TODO: implement
label_names = None  # TODO: implement
deduped = None  # TODO: implement

In [157]:
assert filled["score"].isna().sum()==0
assert abs(filled.iloc[-1]["score"]-.5)<1e-12
assert abs(mean_by_label.loc[0]-.3)<1e-12
assert abs(mean_by_label.loc[1]-((.8+.9+.5)/3))<1e-12
assert count_by_user.loc[1]==2 and count_by_user.loc[2]==1
assert label_names.tolist()==["neg","pos","neg","pos","pos"]
assert len(deduped)<=len(df)
print("✓ pandas drill")

✓ pandas drill


# 27. Hugging Face `datasets` brief review

Core operations:
```python
dataset.map(fn)
dataset.map(fn, batched=True)
dataset.filter(fn)
dataset.select(indices)
dataset.remove_columns(...)
dataset.with_format("torch")
dataset.set_format("torch")
```

Typical tokenization:
```python
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding=False)

tokenized = dataset.map(preprocess, batched=True, remove_columns=["text"])
tokenized = tokenized.with_format("torch")
```

`Dataset.map` transforms examples/batches; pandas `Series.map` is elementwise mapping. `with_format` returns a formatted dataset object; `set_format` changes formatting state.

## 28. Hugging Face mapping drill
Write from memory: batched tokenization of `text`, remove `text`, keep `label`, and format for PyTorch. No automatic test because the notebook should not require a tokenizer download.

In [ ]:
# TODO pseudocode/code:
# def preprocess(batch): ...
# tokenized = dataset.map(...)
# tokenized = tokenized.with_format("torch")

def preprocess(batch):
    # TODO: implement
    pass

tokenized = ds.map(preprocess, batched=True, remove_columns=['text'])
tokenized = tokenized.with_format('torch')